In [2]:
%pip install -qU langchain-community faiss-cpu langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\hema0\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


#### Initialization of embedding model

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\hema0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hema0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hema0\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.c

In [4]:
vector = embeddings.embed_query("Hello world")
print(vector)

[-0.03447727486491203, 0.03102312609553337, 0.006734980270266533, 0.026108933612704277, -0.03936205804347992, -0.16030246019363403, 0.06692394614219666, -0.006441438104957342, -0.047450482845306396, 0.014758863486349583, 0.07087534666061401, 0.05552757531404495, 0.019193356856703758, -0.02625126577913761, -0.01010954286903143, -0.026940442621707916, 0.022307462990283966, -0.02222665585577488, -0.14969263970851898, -0.017493024468421936, 0.007676282897591591, 0.054352231323719025, 0.0032544038258492947, 0.03172588348388672, -0.08462139964103699, -0.029405992478132248, 0.051595550030469894, 0.048124078661203384, -0.003314835485070944, -0.05827915295958519, 0.04196925833821297, 0.022210702300071716, 0.1281888633966446, -0.022338951006531715, -0.011656239628791809, 0.06292837113142014, -0.03287634998559952, -0.09122604131698608, -0.03117534890770912, 0.052699536085128784, 0.04703483358025551, -0.08420310169458389, -0.030056182295084, -0.020744839683175087, 0.009517822414636612, -0.00372176

In [5]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

Explaination for the above code :

faiss → This is the Facebook AI Similarity Search library. It handles fast search of vectors. It only stores and searches numbers (embeddings).

InMemoryDocstore → A simple in-memory store for storing text documents. FAISS cannot store text, so we need this.

FAISS from LangChain → A wrapper class that combines:

FAISS vector index (numbers)

Document store (text)

Mapping from vectors → documents

Embedding function for new texts

In [6]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had bagels , scrambled egg and chai for breakfast.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="It is way too cold upstate New York.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="I am building an AI roadmap as I learn. I am excited!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="The louvre heist was crazy.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="My favourite movie is Avengers",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 cricket players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="How to invest in stock market?.",
    metadata={"source": "tweet"},
)

document_10 = Document(
    page_content="I am way too bored in life",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['279f2451-c5ef-426e-9e5c-01ed04367fb8',
 'a1c579fa-a0b0-4181-82f0-9690180ad79a',
 'd497cdc2-e920-4b32-927a-1ecb1f188208',
 '9c9ebe57-8ba2-4eba-b770-6d9fa8747f35',
 '5d0cac74-db89-4619-94e8-54c7355a7513',
 '2e8c3fb9-a495-49b6-b27a-245cb46350b2',
 '8423c97e-3de4-4282-8b58-e395e7b1108d',
 'd6ba9c31-6040-4dc5-8104-ce516cadcb9f',
 '4baa5185-f3f6-478d-8d09-3987d53d7be9',
 'd0829429-51ac-4924-8fe2-2d6670aa0594']

#### Querying

###### Querying without score

In [7]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": {"$eq": "tweet"}},
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]
* I am building an AI roadmap as I learn. I am excited! [{'source': 'tweet'}]


###### Querying with score

In [8]:
results = vector_store.similarity_search_with_score("Favourite",k=2,filter={"source": "tweet"})

for res , score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=1.207536] My favourite movie is Avengers [{'source': 'tweet'}]
* [SIM=1.717329] I am way too bored in life [{'source': 'tweet'}]


In [9]:
results = vector_store.similarity_search_with_score("heist",k=2,filter={"source": "tweet"})


for res , score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=1.628799] I am way too bored in life [{'source': 'tweet'}]
* [SIM=1.632079] I am building an AI roadmap as I learn. I am excited! [{'source': 'tweet'}]


In [10]:
results = vector_store.similarity_search_with_score("heist",k=2,filter={"source": "news"})

for res , score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=1.131720] The louvre heist was crazy. [{'source': 'news'}]
* [SIM=1.827853] It is way too cold upstate New York. [{'source': 'news'}]


In [11]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=1.248667] It is way too cold upstate New York. [{'source': 'news'}]


#### Making it into a retriver

In [12]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 1})
retriever.invoke("Heist's are a crime", filter={"source": "news"})

[Document(id='9c9ebe57-8ba2-4eba-b770-6d9fa8747f35', metadata={'source': 'news'}, page_content='The louvre heist was crazy.')]

And now we can use the above code to to build a RAG